# Build overnight / term position panels

Identical to `build_main_panel`, but restricts the repo positions to one maturity segment so the position-adjustment impulse responses can be drawn separately for overnight and term repo.

**Usage:** set `SEG` below to `"overnight"` or `"term"`, run all cells, and it writes `Data/monetary_policy_induced_position_<SEG>.csv`. Run once per segment, then run `analysis/position_irf_momentum.do` to produce Figure 5 (`IR_on_momentum`) and the appendix term figure (`IR_term_momentum`).

In [67]:
# === choose the repo maturity segment, then run all cells ===
# run once with SEG = "overnight", once with SEG = "term"
SEG = "term"            # "overnight" -> contractual_maturity <= 1
                             # "term"      -> contractual_maturity > 1
MATURITY_FILTER = "AND contractual_maturity <= 1" if SEG == "overnight" else "AND contractual_maturity > 1"

In [68]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

# Prep

Load packages and open the SFTDS database connection. The hedge-fund and ISIN universes are sourced from `Data/sftds_hedgefunds.csv` and `Data/EA_ISINs.xlsx` (or queried live from the DB).

# 1. Build the main panel

Construct the daily bond × hedge-fund-positioning panel and write `Data/monetary_policy_induced_position.csv` — the input consumed by every analysis `.do` file.

In [69]:
df = pd.read_csv('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\bond_timeseries_v2.csv')

In [70]:
df['Dates'] = pd.to_datetime(df['Dates'])
df['bond_maturity'] = pd.to_datetime(df['bond_maturity'])

C:\Users\hermesf\AppData\Local\Temp\ipykernel_41812\996206371.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['bond_maturity'] = pd.to_datetime(df['bond_maturity'])


In [71]:
# calculate residual maturity in years
df['residual_bond_maturity'] = ((df['bond_maturity'] - df['Dates']).dt.days / 365)

In [72]:
df = df[df['residual_bond_maturity'] >= 0]

In [73]:
df['amt_issued'] = (
    df['amt_issued']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace(' ', '', regex=False)
)

# convert to float, coercing invalid entries to NaN
df['amt_issued'] = pd.to_numeric(df['amt_issued'], errors='coerce')
df['amt_issued'] = df['amt_issued']/1e9

In [74]:
df['amt_issued'].fillna(df['amt_issued'].median(), inplace=True)

In [75]:
df['collateral_country'] = df['ISIN'].str[:2]

In [76]:
df = df[df['collateral_country'].isin(['DE','IT'])]

In [77]:
securities = tuple(df['ISIN'].unique())

In [78]:
# Data prep
query = f""" 

SELECT DISTINCT lender_id

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND lender_country_residence = 'KY' AND s_lender.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
""" 

df_hf = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_41812\3955607767.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_hf = pd.read_sql_query(query, cnxn)


In [79]:
# Data prep
query = f""" 

SELECT DISTINCT borrower_id

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND borrower_country_residence = 'KY' AND s_borrower.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
""" 

df_hf_b = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_41812\1803038206.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_hf_b = pd.read_sql_query(query, cnxn)


In [80]:
# Data prep
query = f""" 

SELECT s.business_date, 
security_isin as isin,
sum(nominal_euro) as borrowing_volume,
avg(contractual_maturity) as long_maturity,
avg(haircut) as long_haircut

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND borrower_country_residence = 'KY' AND s_borrower.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
{MATURITY_FILTER}
GROUP BY business_date, isin
ORDER BY business_date, isin 

""" 

df_borrowing = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_41812\3034409285.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_borrowing = pd.read_sql_query(query, cnxn)


In [81]:
# Data prep
query = f""" 
 

SELECT s.business_date,
security_isin as isin,
sum(nominal_euro) as lending_volume,
avg(contractual_maturity) as short_maturity,
avg(haircut) as short_haircut

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01'  
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND lender_country_residence = 'KY' AND s_lender.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
{MATURITY_FILTER}
GROUP BY business_date, isin
ORDER BY business_date, isin 

""" 

df_lending = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_41812\2505341236.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lending = pd.read_sql_query(query, cnxn)


In [82]:
df_repo = df_borrowing.merge( df_lending, on=['business_date', 'isin'], how='outer', suffixes=('_borrowing', '_lending') )
 

In [83]:
df_repo['business_date'] = pd.to_datetime(df_repo['business_date'])

In [84]:
df = (
    df[['Dates', 'ISIN', 'PX_ASK', 'PX_BID', 'YLD_YTM_ASK', 'YLD_YTM_BID', 'amt_issued', 'residual_bond_maturity', 'collateral_country']]
    .merge(df_repo, right_on=['business_date','isin'], left_on = ['Dates', 'ISIN'], how='left')
)

In [85]:
df = df[df['PX_ASK'].isna() == False]

In [86]:
df.drop(columns=['business_date', 'isin'], inplace=True)

In [87]:
df = df.rename(columns={'Dates': 'business_date'})
df = df.rename(columns={'ISIN': 'isin'})

In [88]:
df['borrowing_volume'].fillna(0, inplace = True)
df['lending_volume'].fillna(0, inplace = True)

In [89]:
df['net_pos'] = (df['borrowing_volume'] - df['lending_volume'])/1e9

In [90]:
shocks_all = pd.read_excel('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\altavilla.xlsx', sheet_name = 'Monetary Event Window')

In [91]:
shocks_all['date'] = pd.to_datetime(shocks_all['date'], dayfirst=True)

In [92]:
out = (
    df.merge(shocks_all[['date', 'OIS_2Y']], left_on=['business_date'], right_on=['date'], how='left')
       .drop(columns=['date'])  # helpers
)


In [93]:
out['OIS_2Y'].fillna(0, inplace=True)

In [94]:
out = out.sort_values(['isin','business_date'])

# mid yield
out['yld_mid'] = (out['YLD_YTM_BID'] + out['YLD_YTM_ASK']) / 2

# ISIN-level daily change
out['delta_y'] = out.groupby('isin')['yld_mid'].diff()

In [95]:
# 1. Base Intensity: Rolling mean of |net_pos| (Magnitude only)
out['abs_net'] = (out['net_pos'].abs() / out['amt_issued']) * 100

In [96]:
hf_roll = (
    out.groupby('isin')['abs_net']
     .apply(lambda s: s.rolling(window=5, min_periods=3).mean().shift(1))
     .reset_index(level=0, drop=True)   # <-- align index with d
)

out['hf_intensity_pre'] = hf_roll

In [97]:
out['net_pos_scaled'] = (out['net_pos'] / out['amt_issued']) * 100

hf_roll_signed = (
    out.groupby('isin')['net_pos_scaled']
     .apply(lambda s: s.rolling(window=5, min_periods=3).mean().shift(1))
     .reset_index(level=0, drop=True)
)

out['hf_intensity_long'] = np.where(hf_roll_signed > 0, out['hf_intensity_pre'], 0)
out['hf_intensity_short'] = np.where(hf_roll_signed < 0, out['hf_intensity_pre'], 0)

In [98]:
out['hf_intensity_pre']= out['hf_intensity_pre'].fillna(0)

In [99]:
out['bid_ask_spread'] = out['PX_ASK'] - out['PX_BID'] 

In [100]:
out['delta_y'] = out['delta_y']*100

In [101]:
ctd = pd.read_excel('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\CTD.xlsx')

In [102]:
# month-end of (Year, Month)
ctd['period_end'] = pd.to_datetime(dict(year=ctd['Year'], month=ctd['Month'], day=1)) + pd.offsets.MonthEnd(0)
ctd['period_start'] = ctd['period_end'] - pd.DateOffset(years=1)

In [103]:
# --- expand & mark in-window ---
m = out.merge(ctd[['isin','period_start','period_end']], on='isin', how='left')
m['in_window'] = (m['business_date'] >= m['period_start']) & (m['business_date'] <= m['period_end'])

# collapse back to one row per (ISIN, business_date): 1 if any window matches
flag_df = (m.groupby(['isin','business_date'], as_index=False)['in_window']
             .any()
             .rename(columns={'in_window':'ctd_flag'}))

# --- attach flag to original `out` ---
out = out.merge(flag_df, on=['isin','business_date'], how='left')
out['ctd_flag'] = out['ctd_flag'].fillna(False).astype(int)


In [104]:
# create binary
out.loc[(out['hf_intensity_pre'] > 0), 'hf_involved'] = 1
out['hf_involved'].fillna(0, inplace = True)

In [105]:
out['prev_net_pos'] = out.groupby('isin')['net_pos'].shift(1)

In [106]:
# create binary
out.loc[(out['prev_net_pos'] > 0), 'hf_involved_long'] = 1
out['hf_involved_long'].fillna(0, inplace = True)

out.loc[(out['prev_net_pos'] < 0), 'hf_involved_short'] = 1
out['hf_involved_short'].fillna(0, inplace = True)

In [107]:
out = out[out['hf_intensity_pre'] < 18]

In [108]:
# Four categories: None / Low / Medium / High
out['hf_category'] = 'ANone'

# Identify ISIN–days with positive HF activity
mask = out['hf_intensity_pre'].fillna(0) > 0

# Within each country, split only those into terciles
out.loc[mask, 'hf_category'] = (
    out.loc[mask]
       .groupby('collateral_country')['hf_intensity_pre']
       .transform(lambda x: pd.qcut(x, 2, labels=['Low', 'High']))
       .astype(str)
)

In [109]:
# Four categories: None / Low / Medium / High
out['hf_category_ext'] = 'ANone'

# Identify ISIN–days with positive HF activity
mask = out['hf_intensity_pre'].fillna(0) > 0

# Within each country, split only those into terciles
out.loc[mask, 'hf_category_ext'] = (
    out.loc[mask]
       .groupby('collateral_country')['hf_intensity_pre']
       .transform(lambda x: pd.qcut(x, 3, labels=['Low', 'Medium', 'High']))
       .astype(str)
)

In [110]:
out = out[out['delta_y'].isna() == False]

In [111]:
out = out[(out['delta_y'] <= 40) & (out['delta_y'] >= -40)]

In [112]:
germany = pd.read_csv(r'C:\Users\hermesf\Projects\JobMarket\Data\NelsonSiegel_DE_prices.csv')

In [113]:
italy = pd.read_csv(r'C:\Users\hermesf\Projects\JobMarket\Data\NelsonSiegel_IT_prices.csv')

In [114]:
spain = pd.read_csv(r'C:\Users\hermesf\Projects\JobMarket\Data\NelsonSiegel_ES_prices.csv')

In [115]:
france = pd.read_csv(r'C:\Users\hermesf\Projects\JobMarket\Data\NelsonSiegel_FR_prices.csv')

In [116]:
df_duration = pd.concat([germany, italy], ignore_index=True)

In [117]:
df_duration['refdate'] = pd.to_datetime(df_duration['refdate'])

In [118]:
out = out.merge(df_duration[['refdate', 'bondcode', 'duration']], left_on = ['business_date', 'isin'], right_on = ['refdate', 'bondcode'], how = 'left')

In [119]:
out['duration'].fillna(out['residual_bond_maturity'], inplace = True)

In [120]:
jk_shock = pd.read_csv('C:\\Users\\hermesf\\Projects\\JobMarket\\Data\\shocks_ecb_mpd_me_d.csv')

In [121]:
jk_shock['business_date'] = pd.to_datetime(jk_shock['date'])

In [122]:
out = out.merge(jk_shock, on = 'business_date', how = 'left')

In [123]:
# Option 1: reassign
out[['pc1', 'STOXX50', 'MP_pm', 'CBI_pm', 'MP_median', 'CBI_median']] = out[['pc1', 'STOXX50', 'MP_pm', 'CBI_pm', 'MP_median', 'CBI_median']].fillna(0)

In [124]:
out[['pc1', 'STOXX50', 'MP_pm', 'CBI_pm', 'MP_median', 'CBI_median']] = out[['pc1', 'STOXX50', 'MP_pm', 'CBI_pm', 'MP_median', 'CBI_median']] * 100

In [125]:
# 1. Calculate the daily change in position (Flow)
out['daily_net_change'] = out.groupby('isin')['net_pos'].diff()
out['delta_intensity'] = (out['daily_net_change'] / out['amt_issued']) * 100

In [126]:
# 3. Determine the Direction ENTERING the shock (State) - Keep this as is
out['is_long_pre'] = (out.groupby('isin')['net_pos'].shift(1) > 0).astype(int)
out['is_short_pre'] = (out.groupby('isin')['net_pos'].shift(1) < 0).astype(int)

In [127]:
out['placebo_shock'] = out.groupby('isin')['OIS_2Y'].shift(15)
out['placebo_shock'].fillna(0, inplace=True)

In [128]:
# Data prep: fund-level LONG positions (HF is the borrower -> long the bond)
query = f""" 

SELECT s.business_date, 
security_isin as isin,
borrower_id as fund_id,
sum(nominal_euro) as long_vol

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND borrower_country_residence = 'KY' AND s_borrower.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
GROUP BY business_date, isin, borrower_id
ORDER BY business_date, isin 

""" 

df_long_fund = pd.read_sql_query(query, cnxn)
df_long_fund['business_date'] = pd.to_datetime(df_long_fund['business_date'])

C:\Users\hermesf\AppData\Local\Temp\ipykernel_41812\85804849.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_long_fund = pd.read_sql_query(query, cnxn)


In [129]:
# Data prep: fund-level SHORT positions (HF is the lender -> short the bond)
query = f""" 

SELECT s.business_date,
security_isin as isin,
lender_id as fund_id,
sum(nominal_euro) as short_vol

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state_backup s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01'  
AND nominal_ccy IN ('EUR') 
AND central_clearing = 'non-cleared' 
AND lender_country_residence = 'KY' AND s_lender.sector = 'IF'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
GROUP BY business_date, isin, lender_id
ORDER BY business_date, isin 

""" 

df_short_fund = pd.read_sql_query(query, cnxn)
df_short_fund['business_date'] = pd.to_datetime(df_short_fund['business_date'])

C:\Users\hermesf\AppData\Local\Temp\ipykernel_41812\3305751002.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_short_fund = pd.read_sql_query(query, cnxn)


In [130]:
# --- Fund-level directionality (duration-weighted) -> holder-weighted bond measure ---

# 1. One row per fund-bond-day with signed net (long +, short -)
fb = df_long_fund.merge(df_short_fund, on=['business_date','isin','fund_id'], how='outer')
fb[['long_vol','short_vol']] = fb[['long_vol','short_vol']].fillna(0)
fb['net']    = fb['long_vol'] - fb['short_vol']
fb['absnet'] = fb['net'].abs()

# 2. fund_dir per fund-day: DV01 (duration-weighted) directionality across ALL bonds
#    |sum net*dur| / sum|net*dur|.  0 = duration-hedged (carry or curve), 1 = directional.
fb = fb.merge(df_duration[['refdate','bondcode','duration']],
              left_on=['business_date','isin'], right_on=['refdate','bondcode'], how='left')
fb = fb.drop(columns=['refdate','bondcode'])
fb['dv01']    = fb['net'] * fb['duration']
fb['absdv01'] = fb['dv01'].abs()
fd = fb.groupby(['business_date','fund_id']).agg(
    net_dv01  =('dv01','sum'),
    gross_dv01=('absdv01','sum'),
).reset_index()
fd['fund_dir'] = fd['net_dv01'].abs() / fd['gross_dv01'].where(fd['gross_dv01'] > 0)
fund_dir = fd[['business_date','fund_id','fund_dir']]

# 3. holder_dir per bond-day: |net|-weighted average of holders' fund_dir.
#    Funds with undefined fund_dir (no duration) drop out of both num and den.
fb = fb.merge(fund_dir, on=['business_date','fund_id'], how='left')
fb['w_num']   = fb['absnet'] * fb['fund_dir']
fb['w_den']   = fb['absnet'].where(fb['fund_dir'].notna(), 0.0)
fb['islong']  = fb['long_vol']  > 0
fb['isshort'] = fb['short_vol'] > 0
holder = fb.groupby(['business_date','isin']).agg(
    holder_num =('w_num','sum'),
    holder_den =('w_den','sum'),
    gross_long =('long_vol','sum'),
    gross_short=('short_vol','sum'),
    n_long     =('islong','sum'),
    n_short    =('isshort','sum'),
).reset_index()
# safe divide: den<=0 becomes NaN so holder_dir is NaN there (stays float)
holder['holder_dir'] = holder['holder_num'] / holder['holder_den'].where(holder['holder_den'] > 0)
holder = holder.drop(columns=['holder_num','holder_den'])

# 4. merge onto out (holder_dir stays NaN where no HF holds the bond = zero-intensity baseline)
out = out.merge(holder, on=['business_date','isin'], how='left')


In [131]:
out = out[['business_date', 'isin', 'PX_ASK', 'PX_BID', 'YLD_YTM_ASK',
       'YLD_YTM_BID', 'amt_issued', 'residual_bond_maturity',
       'collateral_country', 'borrowing_volume', 'long_maturity',
       'long_haircut', 'lending_volume', 'short_maturity', 'short_haircut',
       'net_pos', 'OIS_2Y', 'yld_mid', 'delta_y', 'abs_net',
       'hf_intensity_pre', 'net_pos_scaled', 'hf_intensity_long',
       'hf_intensity_short', 'bid_ask_spread', 'ctd_flag', 'hf_involved',
       'prev_net_pos', 'hf_involved_long', 'hf_involved_short', 'hf_category',
       'hf_category_ext', 'refdate', 'duration', 'date', 'pc1',
       'STOXX50', 'MP_pm', 'CBI_pm', 'MP_median', 'CBI_median',
       'daily_net_change', 'delta_intensity', 'is_long_pre', 'is_short_pre',
       'placebo_shock', 'gross_long', 'n_long', 'gross_short', 'n_short', 'holder_dir']]

In [132]:
out.to_csv(rf'C:\Users\hermesf\Projects\JobMarket\Data\monetary_policy_induced_position_{SEG}.csv')